# **1. Notebook Setup**

## **1.1 Imports**

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf


## **1.2 Notebook Configuration**

In [2]:
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

## **1.3 Paths**

In [3]:
PROJECT_ROOT = Path().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

SRC_DIR = PROJECT_ROOT / "src"

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

# **2. Test data pull (yfinance)**

In [4]:
from src.yfinance_loader import fetch_multiple_companies

### **2.1 Sample tickers**

In [5]:
TICKERS = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "META"
]

SLEEP_SECONDS = 1

In [6]:
fetch_log = fetch_multiple_companies(
    tickers=TICKERS,
    output_dir=RAW_DATA_DIR / "yfinance",
    sleep_seconds=SLEEP_SECONDS
)

fetch_log.head()

Fetching AAPL...


Fetching MSFT...


Fetching GOOGL...


Fetching AMZN...


Fetching META...


,ticker,status,error
0,AAPL,success,None
1,MSFT,success,None
2,GOOGL,success,None
3,AMZN,success,None
4,META,success,None


Everything looks fine for the sample list

# **3. Data pull and Processing**

### **3.1 Pull Universe of Tickers**

In [ ]:
import signal

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

In [7]:
from src.data.ticker_universe import build_us_equity_universe

us_equity_universe = build_us_equity_universe(
    remove_non_operating_securities=True,
    output_path=RAW_DATA_DIR / "ticker_universe" / "us_equity_universe.xlsx",
)

us_equity_universe.head(10)

,Symbol,Company Name,Exchange
0,ACCS,ACCESS Newswire Inc. Common Stock,AMEX
1,AEON,"AEON Biopharma, Inc. Class A Common Stock",AMEX
2,AGIG,Abundia Global Impact Group Inc. Common stock,AMEX
3,AIB,"BlockchAIn Digital Infrastructure, Inc Common ...",AMEX
4,AIRI,Air Industries Group Common Stock,AMEX
5,AMBO,Ambow Education Holding Ltd. American Deposito...,AMEX
6,AMS,American Shared Hospital Services Common Stock,AMEX
7,AMZE,"Amaze Holdings, Inc. Common Stock",AMEX
8,APT,"Alpha Pro Tech, Ltd. Common Stock",AMEX
9,APUS,"Apimeds Pharmaceuticals US, Inc. Common Stock",AMEX


In [8]:
us_equity_universe["Exchange"].value_counts()

Exchange
NASDAQ    3308
NYSE      1857
AMEX       239
Name: count, dtype: int64

Number of company tickers within the ballpark. Good to preceed

### **3.2 Pull Financials based on Ticker Universe**

In [16]:
us_equity_universe = us_equity_universe[
    ~us_equity_universe["Symbol"].str.contains(r"[.$^]", regex=True, na=False)
].copy()

from src.data.financials_loader import (
    fetch_and_flatten_financials,
    filter_companies_with_positive_revenue,
)

In [17]:
financials_flat_df, financials_fetch_log = fetch_and_flatten_financials(
    ticker_universe=us_equity_universe,
    output_dir=RAW_DATA_DIR / "yfinance_financials",
    sleep_seconds=0.75,
    save_every=100,
)

financials_flat_df.head()

Fetching financial statements:   0%|          | 0/5355 [00:00<?, ?it/s]

In [ ]:
financials_fetch_log["Status"].value_counts()

### **3. Filter and Save**

In [ ]:
filtered_financials_df = filter_companies_with_positive_revenue(
    financials_df=financials_flat_df,
    revenue_column="Income_Total Revenue",
)

filtered_financials_df.to_excel(
    INTERIM_DATA_DIR / "financials_positive_revenue.xlsx",
    index=False
)

print(f"Original companies: {len(financials_flat_df):,}")
print(f"Companies with positive revenue: {len(filtered_financials_df):,}")

filtered_financials_df.head()